# Step 6 — Drafting + Grounding Verifier (no hallucinated numbers)

Generate the **script + deck outline + Q&A cheat sheet** with numbers **slotted** (`{{F-00xx}}`),
then **verify** every rendered number against the Fact Store (`docs/grounding.md`). The verifier
blocks any ungrounded number — demonstrated by injecting a hallucination.

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root()
sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print("backends -> qdrant:", settings.qdrant_mode, "| embeddings:", settings.embedding_backend,
      "| sentiment:", settings.sentiment_backend, "| llm:", settings.llm_backend)

backends -> qdrant: memory | embeddings: hash | sentiment: lexicon | llm: mock


In [2]:
from ir_copilot.facts import build_fact_store
from ir_copilot.embeddings import get_embedder
from ir_copilot.vectorstore import WikiStore, WikiChunk
from ir_copilot import corpus
from ir_copilot.agents.sentiment import analyze_sentiment
from ir_copilot.agents.competitor import compare
from ir_copilot.agents.predictive import predict_questions
from ir_copilot.agents.drafting import draft, render_bundle
from ir_copilot.agents.verify import verify

store = build_fact_store(settings.ticker, settings.period, use_mock=settings.use_mock_data)
peers = {p: build_fact_store(p, settings.period, use_mock=settings.use_mock_data) for p in settings.peers}
wiki = WikiStore(get_embedder()); wiki.ensure_collection(recreate=True)
wiki.upsert([WikiChunk(chunk_id=str(i), **c) for i, c in enumerate(corpus.TRANSCRIPT_CHUNKS)])
items = [it for it in (corpus.NEWS_HEADLINES + corpus.SOCIAL_POSTS) if it["ticker"] == settings.ticker]
snap = analyze_sentiment(settings.ticker, items)
pc = compare(store, peers)
qs = predict_questions(store, snap, pc, wiki=wiki)

bundle = draft(store, snap, pc, qs)
rendered = render_bundle(bundle, store)

print("=== SCRIPT (slots rendered to verified values) ===")
for sec in rendered["script"]:
    print(f"\n## {sec['heading']}\n{sec['text']}")
print("\n=== DECK OUTLINE ===")
for sl in rendered["deck_outline"]:
    print(f"\n# {sl['title']}")
    for b in sl["bullets"]:
        print(f"  - {b}")
print("\n=== Q&A CHEAT SHEET ===")
for qa in rendered["qa_cheat_sheet"]:
    print(f"\nQ: {qa['question']}\nA: {qa['suggested_answer']}")

=== SCRIPT (slots rendered to verified values) ===

## Opening
Thank you for joining our FY2026Q2 earnings call. We delivered revenue of $81.61 billion with TTM EPS of $6.53, reflecting continued execution.

## Profitability
Gross margin was 74.9% and operating margin was 65.6%, supported by strong return on equity of 33.1% and ROIC of 31.4%.

## Competitive position
We continue to lead our peer group on gross_margin, operating_margin, roe, roic, roa, ttm_eps. Our market capitalization stands at $5.04 trillion.

=== DECK OUTLINE ===

# Financial Highlights
  - Revenue $81.61 billion
  - TTM EPS $6.53
  - Gross margin 74.9%
  - Operating margin 65.6%

# Returns & Valuation
  - ROE 33.1%
  - ROIC 31.4%
  - Market cap $5.04 trillion
  - TTM PE 31.9x

# Competitive Position
  - gross margin: 74.9% — leads peer set
  - operating margin: 65.6% — leads peer set
  - roe: 33.1% — leads peer set
  - roic: 31.4% — leads peer set
  - roa: 25.0% — leads peer set
  - ttm eps: $6.53 — leads peer set


/Users/v843010/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Grounding verifier — clean vs hallucinated

In [3]:
rep = verify(rendered, store)
print("clean draft  -> passed:", rep.passed, "| violations:", rep.numeric_violations,
      "| missing:", rep.coverage_missing)

# Inject a hallucinated figure into a Q&A answer -> must be caught.
import copy
bad = copy.deepcopy(rendered)
bad["qa_cheat_sheet"][0]["suggested_answer"] += " Actually EPS was 9.99 dollars."
rep_bad = verify(bad, store)
print("injected 9.99 -> passed:", rep_bad.passed, "| violations:", rep_bad.numeric_violations)

assert rep.passed and not rep_bad.passed and 9.99 in rep_bad.numeric_violations
print("\nGROUNDING ENFORCED: clean draft passes; hallucinated 9.99 is blocked.")

clean draft  -> passed: True | violations: [] | missing: []
injected 9.99 -> passed: False | violations: [9.99]

GROUNDING ENFORCED: clean draft passes; hallucinated 9.99 is blocked.


**Next (Step 7):** wire all agents into the LangGraph orchestrator with the human-in-the-loop gate.